# **CTDDG Evaluation Metrics**

## Configuration

In [1]:

# ============================================================
# NOTEBOOK 3 — BLOCK 1
# CONFIGURATION FOR FIVE-MODEL EVALUATION
# ============================================================

from pathlib import Path
from collections import OrderedDict
import pandas as pd
import numpy as np
import json
import hashlib
import sys

PROJECT_ROOT = Path("/home/muhamed/repo/CTDDG")
REWORKED_ROOT = PROJECT_ROOT / "REWORKED_CODE"
EVALUATION_ROOT = REWORKED_ROOT / "EVALUATION_OUTPUT"

EMBEDDINGS_ROOT = EVALUATION_ROOT / "embeddings"
GENERATED_ROOT = EVALUATION_ROOT / "generated_samples"

# Change to "all5" after the other two models finish.
EVALUATION_MODE = "phase1"

assert EVALUATION_MODE in {"phase1", "all5"}

EXPECTED_RUNS = 10
EXPECTED_PER_RUN = 100
EXPECTED_PER_SLOT = EXPECTED_RUNS * EXPECTED_PER_RUN

# ------------------------------------------------------------
# A. EXACT MODEL SOURCES
# ------------------------------------------------------------

MODEL_SPECS = OrderedDict({
    "finetuned_cdgcn_to_ctddg": {
        "phase": 1,
        "conditioned": True,
        "checkpoint": REWORKED_ROOT / (
            "finetuning/finetuning_runs/"
            "cdgcn_to_ctddg_d1_bs32_lambda0p1_v1/"
            "ckpt_best.pt"
        ),
        "vocab": REWORKED_ROOT / (
            "chembl_preprocessed/atom_types.txt"
        ),
        "atom_types": 43,
    },

    "pretrained_no_toxicity": {
        "phase": 1,
        "conditioned": False,
        "checkpoint": REWORKED_ROOT / (
            "PRETRAINING_OUTPUT/no_toxicity/training_runs/"
            "chembl_B32_K5_alpha08_seed42/best.pt"
        ),
        "vocab": REWORKED_ROOT / (
            "PREPROCESSING_OUTPUT/vocabulary/atom_types.txt"
        ),
        "atom_types": 55,
    },

    "pretrained_with_toxicity": {
        "phase": 1,
        "conditioned": False,
        "checkpoint": REWORKED_ROOT / (
            "PRETRAINING_OUTPUT/toxicity_aware/runs/"
            "ctddg_main_bs32_lambda2/model_best.pt"
        ),
        "vocab": REWORKED_ROOT / (
            "PREPROCESSING_OUTPUT/vocabulary/atom_types.txt"
        ),
        "atom_types": 55,
    },

    "older_pretrained_no_toxicity": {
        "phase": 2,
        "conditioned": False,
        "checkpoint": REWORKED_ROOT / (
            "pretraining/pretraining_runs/"
            "cdgcn_torch_nll_bs32_v1/"
            "model_weights_final.pt"
        ),
        "vocab": REWORKED_ROOT / (
            "chembl_preprocessed/atom_types.txt"
        ),
        "atom_types": 43,
    },

    "older_pretrained_with_toxicity": {
        "phase": 2,
        "conditioned": False,
        "checkpoint": REWORKED_ROOT / (
            "pretraining/pretraining_runs/"
            "ctddg_torch_eq8_bs32_v1/checkpoint.pt"
        ),
        "vocab": REWORKED_ROOT / (
            "chembl_preprocessed/atom_types.txt"
        ),
        "atom_types": 43,
    },
})

MODEL_INPUT_ROOTS = {
    1: GENERATED_ROOT / "phase1_v1",
    2: GENERATED_ROOT / "phase2_v1",
}

ACTIVE_MODELS = [
    name for name, spec in MODEL_SPECS.items()
    if EVALUATION_MODE == "all5" or spec["phase"] == 1
]

RESULTS_DIR = (
    EVALUATION_ROOT / "metrics"
    / f"{EVALUATION_MODE}_evaluation_v1"
)

# ------------------------------------------------------------
# B. EVALUATION SETTINGS
# ------------------------------------------------------------

# Novelty requires a verified training/reference dataset.
NOVELTY_REFERENCE_BY_MODEL = {}

# Same Ames RF model for every model.
RF_MODEL_PATH = (
    REWORKED_ROOT / "Random_Forest/RF_NItesh.joblib"
)

# These are candidate settings, not yet verified.
RF_RADIUS = 2
RF_BITS = 2048
RF_POSITIVE_LABEL = 1
RF_FEATURE_CONTRACT_VERIFIED = False

# Diversity will use a fixed number of molecule pairs,
# avoiding expensive all-against-all comparisons.
DIVERSITY_MAX_PAIRS = 5000
EVALUATION_SEED = 42

# ------------------------------------------------------------
# C. LOAD VERIFIED PROTEIN MANIFESTS
# ------------------------------------------------------------

FOLD_MANIFESTS = {}

for fold in ["1002_103", "1042_104"]:

    manifest_path = (
        EMBEDDINGS_ROOT / fold / "protein_manifest.csv"
    )

    assert manifest_path.is_file(), manifest_path

    manifest = pd.read_csv(manifest_path)

    required = {
        "protein_id", "sequence_sha256",
        "cleaned_order_id"
    }

    assert required.issubset(manifest.columns)
    assert manifest.protein_id.is_unique

    expected_ids = [
        f"protein{i}" for i in range(1, len(manifest) + 1)
    ]

    assert manifest.protein_id.tolist() == expected_ids

    FOLD_MANIFESTS[fold] = manifest

EXPECTED_SLOTS_PER_MODEL = sum(
    len(manifest) for manifest in FOLD_MANIFESTS.values()
)

# ------------------------------------------------------------
# D. VERIFY MODEL FILES
# ------------------------------------------------------------

for name in ACTIVE_MODELS:

    spec = MODEL_SPECS[name]

    assert spec["checkpoint"].is_file(), (
        name, spec["checkpoint"]
    )
    assert spec["vocab"].is_file(), spec["vocab"]

    vocab_count = len([
        line for line in spec["vocab"].read_text().splitlines()
        if line.strip()
    ])

    assert vocab_count == spec["atom_types"]

print("=" * 65)
print("NOTEBOOK 3 — CONFIGURATION")
print("=" * 65)

print("Mode:", EVALUATION_MODE)
print("Models:", ACTIVE_MODELS)
print("Folds:", {
    fold: len(m)
    for fold, m in FOLD_MANIFESTS.items()
})
print("Slots per model:", EXPECTED_SLOTS_PER_MODEL)
print("Runs per slot:", EXPECTED_RUNS)
print("Samples per run:", EXPECTED_PER_RUN)

print(
    "Expected accepted samples:",
    EXPECTED_SLOTS_PER_MODEL
    * EXPECTED_PER_SLOT
    * len(ACTIVE_MODELS)
)

print("Evaluation output:", RESULTS_DIR)
print("\nBLOCK 1 PASSED — no files modified.")


NOTEBOOK 3 — CONFIGURATION
Mode: phase1
Models: ['finetuned_cdgcn_to_ctddg', 'pretrained_no_toxicity', 'pretrained_with_toxicity']
Folds: {'1002_103': 103, '1042_104': 99}
Slots per model: 202
Runs per slot: 10
Samples per run: 100
Expected accepted samples: 606000
Evaluation output: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/metrics/phase1_evaluation_v1

BLOCK 1 PASSED — no files modified.


In [2]:

# ============================================================
# NOTEBOOK 3 — BLOCK 2
# READ-ONLY GENERATED SAMPLE INVENTORY
# ============================================================

import json
import pandas as pd

inventory_rows = []

for model_name in ACTIVE_MODELS:

    spec = MODEL_SPECS[model_name]

    model_root = (
        MODEL_INPUT_ROOTS[spec["phase"]] / model_name
    )

    for fold, manifest in FOLD_MANIFESTS.items():

        for row in manifest.itertuples(index=False):

            protein_id = row.protein_id

            folder = model_root / fold / protein_id

            main_path = (
                folder / f"{protein_id}_all_samples.csv"
            )

            raw_path = (
                folder / f"{protein_id}_raw_attempts.csv"
            )

            metadata_path = folder / "generation_info.json"

            completed_run_metadata = sum(
                (
                    folder / "runs"
                    / f"run_{run_id:02d}"
                    / "run_info.json"
                ).is_file()
                for run_id in range(1, EXPECTED_RUNS + 1)
            )

            main_exists = main_path.is_file()
            raw_exists = raw_path.is_file()
            metadata_exists = metadata_path.is_file()

            if metadata_exists:

                metadata = json.loads(
                    metadata_path.read_text()
                )

                # Check model and protein identity before
                # accepting the saved metadata.
                assert metadata["model"] == model_name
                assert metadata["fold"] == fold
                assert metadata["protein_id"] == protein_id

                assert (
                    metadata["protein_sequence_sha256"]
                    == row.sequence_sha256
                )

                assert (
                    metadata["conditioned_on_protein"]
                    == spec["conditioned"]
                )

                metadata_ready = (
                    metadata.get("runs") == EXPECTED_RUNS
                    and metadata.get("target_per_run")
                        == EXPECTED_PER_RUN
                    and metadata.get("accepted_molecules")
                        == EXPECTED_PER_SLOT
                )

            else:
                metadata_ready = False

            if (
                main_exists
                and raw_exists
                and metadata_ready
                and completed_run_metadata == EXPECTED_RUNS
            ):
                status = "READY_FOR_QA"

            elif (
                main_exists
                or raw_exists
                or metadata_exists
                or completed_run_metadata > 0
            ):
                status = "IN_PROGRESS_OR_PARTIAL"

            else:
                status = "NOT_STARTED"

            inventory_rows.append({
                "model": model_name,
                "fold": fold,
                "protein_id": protein_id,
                "status": status,
                "completed_runs": completed_run_metadata,
                "main_exists": main_exists,
                "raw_exists": raw_exists,
                "metadata_exists": metadata_exists,
                "main_path": str(main_path),
                "raw_path": str(raw_path),
                "metadata_path": str(metadata_path),
            })

INPUT_INVENTORY = pd.DataFrame(inventory_rows)

summary = (
    INPUT_INVENTORY
    .groupby(["model", "fold", "status"])
    .size()
    .unstack(fill_value=0)
)

print("=" * 65)
print("GENERATED SAMPLES — INPUT INVENTORY")
print("=" * 65)

display(summary)

ready = int(
    (INPUT_INVENTORY["status"] == "READY_FOR_QA").sum()
)

total = len(INPUT_INVENTORY)

print(f"\nReady for detailed verification: {ready}/{total}")

if ready == total:
    print(
        "All expected outputs have completion metadata. "
        "Detailed CSV verification is next."
    )
else:
    print(
        "Generation is not fully complete for the selected "
        "models. This inventory can be rerun later."
    )

print("\nBLOCK 2 COMPLETE — read-only inventory.")


GENERATED SAMPLES — INPUT INVENTORY


status                             IN_PROGRESS_OR_PARTIAL  NOT_STARTED  \
model                    fold                                            
finetuned_cdgcn_to_ctddg 1002_103                       0            0   
                         1042_104                       1           52   
pretrained_no_toxicity   1002_103                       0          103   
                         1042_104                       0           99   
pretrained_with_toxicity 1002_103                       0          103   
                         1042_104                       0           99   

status                             READY_FOR_QA  
model                    fold                    
finetuned_cdgcn_to_ctddg 1002_103           103  
                         1042_104            46  
pretrained_no_toxicity   1002_103             0  
                         1042_104             0  
pretrained_with_toxicity 1002_103             0  
                         1042_104             0


Ready for detailed verification: 149/606
Generation is not fully complete for the selected models. This inventory can be rerun later.

BLOCK 2 COMPLETE — read-only inventory.


## Verify Saved Sample Files

In [3]:

# ============================================================
# NOTEBOOK 3 — BLOCK 3
# VERIFY COMPLETED SAMPLE GROUPS
# ============================================================

import hashlib
import json
from collections import Counter
import pandas as pd
import numpy as np
import h5py

# Preview only while generation is running.
# Set to None later to validate ALL completed groups.
QA_MAX_GROUPS_PER_MODEL = 2

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def check(condition, message):
    if not condition:
        raise ValueError(message)

# Work only on groups that Block 2 marked ready.
candidates = INPUT_INVENTORY[
    INPUT_INVENTORY.status == "READY_FOR_QA"
].sort_values(["model", "fold", "protein_id"])

if QA_MAX_GROUPS_PER_MODEL is not None:
    candidates = candidates.groupby(
        "model", group_keys=False
    ).head(QA_MAX_GROUPS_PER_MODEL)

# Hash checkpoints once, not for every protein.
model_hashes = {
    name: {
        "checkpoint": sha256_file(spec["checkpoint"]),
        "vocab": sha256_file(spec["vocab"]),
    }
    for name, spec in MODEL_SPECS.items()
    if name in ACTIVE_MODELS
}

QA_VERIFIED_FILES = []
QA_ERRORS = []

for row in candidates.itertuples(index=False):

    try:
        key = f"{row.model}/{row.fold}/{row.protein_id}"

        folder = (
            Path(row.main_path).parent
        )

        main = pd.read_csv(
            row.main_path, keep_default_na=False
        )
        raw = pd.read_csv(
            row.raw_path, keep_default_na=False
        )
        info = json.loads(
            Path(row.metadata_path).read_text()
        )

        check(
            list(main.columns) == ["Run_ID", "SMILES"],
            f"{key}: invalid accepted CSV columns"
        )

        required_raw = {
            "Run_ID", "Attempt_ID", "SMILES",
            "decode_status", "saved_in_main_csv"
        }
        check(
            required_raw.issubset(raw.columns),
            f"{key}: invalid raw CSV columns"
        )

        check(
            len(main) == EXPECTED_PER_SLOT,
            f"{key}: wrong total accepted count"
        )
        check(
            len(raw) == info["raw_attempts"],
            f"{key}: raw-attempt count mismatch"
        )

        check(
            sha256_file(row.main_path) == info["samples_sha256"],
            f"{key}: accepted CSV checksum mismatch"
        )
        check(
            sha256_file(row.raw_path) == info["raw_attempts_sha256"],
            f"{key}: raw CSV checksum mismatch"
        )

        check(
            info["checkpoint_sha256"]
            == model_hashes[row.model]["checkpoint"],
            f"{key}: wrong checkpoint"
        )
        check(
            info["vocabulary_sha256"]
            == model_hashes[row.model]["vocab"],
            f"{key}: wrong atom vocabulary"
        )

        manifest = FOLD_MANIFESTS[row.fold]
        source = manifest.set_index("protein_id").loc[
            row.protein_id
        ]

        check(
            info["protein_sequence_sha256"]
            == source["sequence_sha256"],
            f"{key}: incorrect protein association"
        )

        conditioned = MODEL_SPECS[row.model]["conditioned"]
        check(
            info["conditioned_on_protein"] is conditioned,
            f"{key}: incorrect conditioning flag"
        )

        if conditioned:
            embedding_path = (
                EMBEDDINGS_ROOT / row.fold
                / "prose_embeddings_avg_6165.h5"
            )
            protein_index = int(source["cleaned_order_id"])

            with h5py.File(embedding_path, "r") as h5:
                vector = np.asarray(
                    h5[f"p{protein_index}"],
                    dtype=np.float32
                )

            check(
                vector.shape == (6165,),
                f"{key}: wrong embedding dimension"
            )

            embedding_hash = hashlib.sha256(
                vector.tobytes()
            ).hexdigest()

            check(
                info["protein_embedding_sha256"]
                == embedding_hash,
                f"{key}: embedding hash mismatch"
            )

        else:
            check(
                info["protein_embedding_sha256"] is None,
                f"{key}: unexpected protein conditioning"
            )

        # Verify every run and its independent snapshot.
        for run_id in range(1, EXPECTED_RUNS + 1):

            a = main[main.Run_ID == run_id]
            r = raw[raw.Run_ID == run_id]

            check(
                len(a) == EXPECTED_PER_RUN,
                f"{key}: run {run_id} wrong accepted count"
            )
            check(
                a.SMILES.nunique() == EXPECTED_PER_RUN,
                f"{key}: run {run_id} not unique"
            )
            check(
                len(r) >= EXPECTED_PER_RUN,
                f"{key}: run {run_id} missing attempts"
            )
            check(
                r.Attempt_ID.tolist()
                == list(range(1, len(r) + 1)),
                f"{key}: run {run_id} invalid attempt IDs"
            )

            flags = (
                r.saved_in_main_csv.astype(str)
                .str.lower().isin(("true", "1"))
            )

            check(
                Counter(r.loc[flags, "SMILES"])
                == Counter(a.SMILES),
                f"{key}: run {run_id} raw/accepted mismatch"
            )

            run_dir = (
                folder / "runs" / f"run_{run_id:02d}"
            )
            run_info = json.loads(
                (run_dir / "run_info.json").read_text()
            )

            check(
                run_info["run_id"] == run_id
                and run_info["accepted"] == EXPECTED_PER_RUN
                and run_info["attempts"] == len(r),
                f"{key}: run metadata inconsistent"
            )

            check(
                sha256_file(run_dir / "accepted.csv")
                == run_info["accepted_sha256"],
                f"{key}: run accepted snapshot corrupted"
            )
            check(
                sha256_file(run_dir / "raw_attempts.csv")
                == run_info["raw_sha256"],
                f"{key}: run raw snapshot corrupted"
            )

        QA_VERIFIED_FILES.append({
            "model": row.model,
            "fold": row.fold,
            "protein_id": row.protein_id,
            "main_path": row.main_path,
            "raw_path": row.raw_path,
            "accepted": len(main),
            "attempts": len(raw),
        })

        print(f"PASS: {key} — {len(main)} accepted")

    except Exception as exc:
        QA_ERRORS.append({
            "model": row.model,
            "fold": row.fold,
            "protein_id": row.protein_id,
            "error": str(exc),
        })

print("\n" + "=" * 65)
print("BLOCK 3 — SAMPLE INTEGRITY RESULTS")
print("=" * 65)

print("Verified groups:", len(QA_VERIFIED_FILES))
print("Failed groups:", len(QA_ERRORS))

if QA_ERRORS:
    display(pd.DataFrame(QA_ERRORS))
    raise RuntimeError("Sample QA failed; inspect errors.")

print("PASS — inspected groups are internally consistent.")
print("This is a preview, NOT full-corpus certification.")


PASS: finetuned_cdgcn_to_ctddg/1002_103/protein1 — 1000 accepted
PASS: finetuned_cdgcn_to_ctddg/1002_103/protein10 — 1000 accepted

BLOCK 3 — SAMPLE INTEGRITY RESULTS
Verified groups: 2
Failed groups: 0
PASS — inspected groups are internally consistent.
This is a preview, NOT full-corpus certification.


## RDKit Metrics: Definitions and Preview

In [4]:

# ============================================================
# NOTEBOOK 3 — BLOCK 4
# CORE METRIC FUNCTIONS + SMALL-SCALE TEST
# ============================================================

import importlib.util
import numpy as np
import pandas as pd

from functools import lru_cache
from rdkit import Chem, RDConfig, RDLogger
from rdkit.Chem import (
    Descriptors, Crippen, Lipinski,
    QED, rdMolDescriptors
)

RDLogger.DisableLog("rdApp.error")

# ------------------------------------------------------------
# A. LOAD OFFICIAL RDKit SAS SCORER
# ------------------------------------------------------------

sas_file = (
    Path(RDConfig.RDContribDir)
    / "SA_Score" / "sascorer.py"
)

assert sas_file.is_file(), (
    f"RDKit SAS implementation missing: {sas_file}"
)

sas_spec = importlib.util.spec_from_file_location(
    "ctddg_sascorer", sas_file
)

sascorer = importlib.util.module_from_spec(sas_spec)
sas_spec.loader.exec_module(sascorer)

print("SAS scoring implementation:", sas_file)

# ------------------------------------------------------------
# B. CHEMICAL VALIDITY
# ------------------------------------------------------------

@lru_cache(maxsize=100000)
def canonicalize_smiles(smiles):

    if not isinstance(smiles, str) or not smiles.strip():
        return None

    try:
        molecule = Chem.MolFromSmiles(smiles)

        if molecule is None or molecule.GetNumAtoms() == 0:
            return None

        return Chem.MolToSmiles(
            molecule,
            canonical=True,
            isomericSmiles=True
        )

    except Exception:
        return None


# ------------------------------------------------------------
# C. MOLECULAR DESCRIPTORS
# ------------------------------------------------------------

@lru_cache(maxsize=100000)
def molecular_properties(canonical_smiles):

    mol = Chem.MolFromSmiles(canonical_smiles)

    assert mol is not None

    return {
        "logP": float(Crippen.MolLogP(mol)),
        "MW": float(Descriptors.ExactMolWt(mol)),
        "HBD": int(Lipinski.NumHDonors(mol)),
        "HBA": int(Lipinski.NumHAcceptors(mol)),
        "RB": int(Lipinski.NumRotatableBonds(mol)),
        "TPSA": float(rdMolDescriptors.CalcTPSA(mol)),
        "QED": float(QED.qed(mol)),
        "SAS": float(sascorer.calculateScore(mol)),
    }


def percentage(numerator, denominator):
    return 100.0 * numerator / denominator if denominator else np.nan


# ------------------------------------------------------------
# D. COMPUTE METRICS FOR ONE PROTEIN/RUN
# ------------------------------------------------------------

def evaluate_protein_run(
    accepted,
    raw_attempts,
    model_name,
    fold,
    protein_id,
    run_id
):

    # Exported molecules
    accepted_canonical = [
        canonicalize_smiles(sm)
        for sm in accepted["SMILES"]
    ]

    valid_canonical = [
        sm for sm in accepted_canonical
        if sm is not None
    ]

    # Every raw attempt stays in the denominator.
    raw_valid_flags = [
        canonicalize_smiles(sm) is not None
        for sm in raw_attempts["SMILES"]
    ]

    raw_ended_valid = sum(
        valid and status == "ended"
        for valid, status in zip(
            raw_valid_flags,
            raw_attempts["decode_status"]
        )
    )

    raw_valid_count = sum(raw_valid_flags)
    exported = len(accepted)
    valid_exported = len(valid_canonical)

    # Basic metrics
    result = {
        "model": model_name,
        "fold": fold,
        "protein_id": protein_id,
        "run_id": run_id,

        "exported_count": exported,
        "raw_attempt_count": len(raw_attempts),

        # Method A: mentor-style exported yield.
        "mentor_validity_pct": percentage(
            exported, EXPECTED_PER_RUN
        ),

        # Method B: independently RDKit-valid raw attempts.
        "raw_rdkit_validity_pct": percentage(
            raw_valid_count, len(raw_attempts)
        ),

        # Stricter complete-and-valid graph diagnostic.
        "raw_ended_valid_pct": percentage(
            raw_ended_valid, len(raw_attempts)
        ),

        "exported_rdkit_validity_pct": percentage(
            valid_exported, exported
        ),

        "mentor_uniqueness_pct": percentage(
            accepted.SMILES.nunique(),
            EXPECTED_PER_RUN
        ),

        "canonical_uniqueness_valid_pct": percentage(
            len(set(valid_canonical)), valid_exported
        ),

        "valid_exported_count": valid_exported,
        "raw_valid_count": raw_valid_count,
    }

    if not valid_canonical:
        return result

    # Calculate descriptors only on valid molecules.
    properties = [
        molecular_properties(sm)
        for sm in valid_canonical
    ]

    props = pd.DataFrame(properties)

    result["QED_mean_valid"] = float(props.QED.mean())
    result["SAS_mean_valid"] = float(props.SAS.mean())

    # Mentor-compatible QED denominator.
    result["mentor_QED"] = (
        float(props.QED.sum()) / EXPECTED_PER_RUN
    )

    unique_exported_raw = accepted.SMILES.nunique()

    threshold_tests = {
        "logP_lt5": props.logP < 5,
        "MW_lt500": props.MW < 500,
        "HBD_lt5": props.HBD < 5,
        "HBA_lt10": props.HBA < 10,
        "RB_lt10": props.RB < 10,
        "TPSA_lt140": props.TPSA < 140,
        "SAS_lt6": props.SAS < 6,
    }

    for metric, passed in threshold_tests.items():

        pass_count = int(passed.sum())

        # Conventional valid-molecule pass fraction.
        result[f"{metric}_standard_pct"] = percentage(
            pass_count, valid_exported
        )

        # Exact mentor reference weighting.
        result[f"{metric}_mentor_pct"] = (
            100
            * (unique_exported_raw / EXPECTED_PER_RUN)
            * (pass_count / EXPECTED_PER_RUN)
        )

    violations = (
        (props.logP >= 5).astype(int)
        + (props.MW >= 500).astype(int)
        + (props.HBD >= 5).astype(int)
        + (props.HBA >= 10).astype(int)
    )

    result["lipinski_zero_violations_pct"] = percentage(
        int((violations == 0).sum()),
        valid_exported
    )

    return result


# ------------------------------------------------------------
# E. TEST USING THE VERIFIED GROUPS
# ------------------------------------------------------------

preview_rows = []

for item in QA_VERIFIED_FILES:

    accepted = pd.read_csv(
        item["main_path"],
        keep_default_na=False
    )

    raw = pd.read_csv(
        item["raw_path"],
        keep_default_na=False
    )

    for run_id in range(1, EXPECTED_RUNS + 1):

        a = accepted[
            accepted.Run_ID == run_id
        ]

        r = raw[
            raw.Run_ID == run_id
        ]

        preview_rows.append(
            evaluate_protein_run(
                a,
                r,
                item["model"],
                item["fold"],
                item["protein_id"],
                run_id
            )
        )

METRIC_PREVIEW = pd.DataFrame(preview_rows)

print("\n" + "=" * 65)
print("BLOCK 4 — METRIC PREVIEW")
print("=" * 65)

print("Protein/run groups evaluated:", len(METRIC_PREVIEW))

if not METRIC_PREVIEW.empty:

    columns = [
        "mentor_validity_pct",
        "raw_rdkit_validity_pct",
        "raw_ended_valid_pct",
        "exported_rdkit_validity_pct",
        "mentor_uniqueness_pct",
        "QED_mean_valid",
        "SAS_mean_valid",
    ]

    display(
        METRIC_PREVIEW.groupby("model")[columns]
        .mean()
        .round(3)
    )

print("\nPreview complete.")
print("No production evaluation files modified.")


SAS scoring implementation: /home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/rdkit/Contrib/SA_Score/sascorer.py

BLOCK 4 — METRIC PREVIEW
Protein/run groups evaluated: 20


,mentor_validity_pct,raw_rdkit_validity_pct,raw_ended_valid_pct,exported_rdkit_validity_pct,mentor_uniqueness_pct,QED_mean_valid,SAS_mean_valid
model,,,,,,,
finetuned_cdgcn_to_ctddg,100.0,95.312,95.312,100.0,100.0,0.463,3.129



Preview complete.
No production evaluation files modified.


## Molecular diversity and novelty

In [5]:

# ============================================================
# NOTEBOOK 3 — BLOCK 5
# MOLECULAR DIVERSITY + NOVELTY
# ============================================================

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from functools import lru_cache
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib

# ------------------------------------------------------------
# A. FINGERPRINT CONFIGURATION
# ------------------------------------------------------------

DIVERSITY_RADIUS = 2
DIVERSITY_BITS = 2048
DIVERSITY_MAX_PAIRS = 5000
DIVERSITY_SEED = 42

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=DIVERSITY_RADIUS,
    fpSize=DIVERSITY_BITS
)

@lru_cache(maxsize=20000)
def get_morgan_fp(smiles):
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise ValueError(f"Invalid canonical SMILES: {smiles}")
    return morgan_generator.GetFingerprint(molecule)

# ------------------------------------------------------------
# B. OPTIONAL VERIFIED NOVELTY REFERENCES
# ------------------------------------------------------------

# Leave empty until we identify the actual training
# ligand corpus for each model.
#
# Example (do not enable without verification):
#
# NOVELTY_REFERENCES = {
#     "model_name": {
#         "path": Path("/verified/training_molecules.csv.gz"),
#         "column": "canonical_smiles",
#         "verified": True,
#         "scope": "full_training_corpus"
#     }
# }
#
# "common_reference" is also an allowed scope.
# It measures novelty to a shared corpus, not
# novelty to a particular model's training data.

NOVELTY_REFERENCES = {}

def load_novelty_reference(config):
    path = Path(config["path"])

    assert config.get("verified") is True
    assert config["scope"] in {
        "full_training_corpus", "common_reference"
    }
    assert path.is_file(), path

    references = set()

    if path.name.endswith((".csv", ".csv.gz", ".tsv")):
        sep = "\t" if path.name.endswith(".tsv") else ","

        for chunk in pd.read_csv(
            path,
            sep=sep,
            usecols=[config["column"]],
            chunksize=20000,
            keep_default_na=False
        ):
            for smiles in chunk[config["column"]]:
                canonical = canonicalize_smiles(str(smiles))
                if canonical is not None:
                    references.add(canonical)

    elif path.suffix.lower() in {".txt", ".smi"}:
        for line in path.read_text().splitlines():
            if not line.strip() or line.startswith("#"):
                continue
            canonical = canonicalize_smiles(
                line.split()[0]
            )
            if canonical is not None:
                references.add(canonical)

    else:
        raise ValueError(f"Unsupported reference: {path}")

    if not references:
        raise ValueError(f"No valid molecules in {path}")

    return references

REFERENCE_SETS = {
    model: load_novelty_reference(config)
    for model, config in NOVELTY_REFERENCES.items()
}

# ------------------------------------------------------------
# C. CALCULATE INTERNAL DIVERSITY
# ------------------------------------------------------------

def calculate_internal_diversity(smiles_list, seed):

    canonical = sorted(set(smiles_list))
    n = len(canonical)

    if n < 2:
        return np.nan, 0

    fingerprints = [
        get_morgan_fp(sm) for sm in canonical
    ]

    total_possible_pairs = n * (n - 1) // 2

    # Exact all-pairs diversity for small groups.
    if total_possible_pairs <= DIVERSITY_MAX_PAIRS:

        similarity_sum = 0.0
        pair_count = 0

        for i in range(n - 1):
            similarities = DataStructs.BulkTanimotoSimilarity(
                fingerprints[i],
                fingerprints[i + 1:]
            )

            similarity_sum += sum(similarities)
            pair_count += len(similarities)

        mean_similarity = similarity_sum / pair_count

        return 1.0 - mean_similarity, pair_count

    # Uniform random sampling of unordered molecule pairs.
    rng = np.random.default_rng(seed)
    pair_count = DIVERSITY_MAX_PAIRS

    similarity_sum = 0.0
    completed = 0

    while completed < pair_count:
        i, j = rng.integers(0, n, size=2)

        if i == j:
            continue

        similarity_sum += DataStructs.TanimotoSimilarity(
            fingerprints[int(i)],
            fingerprints[int(j)]
        )

        completed += 1

    return 1.0 - similarity_sum / pair_count, pair_count

# ------------------------------------------------------------
# D. EVALUATE VERIFIED PREVIEW GROUPS
# ------------------------------------------------------------

diversity_rows = []

for item in QA_VERIFIED_FILES:

    accepted = pd.read_csv(
        item["main_path"],
        keep_default_na=False
    )

    model = item["model"]
    fold = item["fold"]
    protein_id = item["protein_id"]

    reference_set = REFERENCE_SETS.get(model)
    reference_scope = (
        NOVELTY_REFERENCES[model]["scope"]
        if reference_set is not None else "unavailable"
    )

    for run_id in range(1, EXPECTED_RUNS + 1):

        group = accepted[
            accepted["Run_ID"] == run_id
        ]

        canonical = [
            canonicalize_smiles(sm)
            for sm in group["SMILES"]
        ]

        valid = [s for s in canonical if s is not None]
        unique_valid = set(valid)

        group_key = f"{model}|{fold}|{protein_id}|{run_id}"
        seed = int(hashlib.sha256(
            group_key.encode()
        ).hexdigest()[:8], 16) + DIVERSITY_SEED

        diversity, pairs = calculate_internal_diversity(
            list(unique_valid), seed
        )

        novelty = np.nan

        if reference_set is not None and unique_valid:
            novelty = 100 * (
                len(unique_valid - reference_set)
                / len(unique_valid)
            )

        diversity_rows.append({
            "model": model,
            "fold": fold,
            "protein_id": protein_id,
            "run_id": run_id,
            "valid_molecules": len(valid),
            "unique_canonical": len(unique_valid),
            "internal_diversity": diversity,
            "pairs_compared": pairs,
            "novelty_pct": novelty,
            "novelty_reference_scope": reference_scope,
        })

DIVERSITY_PREVIEW = pd.DataFrame(diversity_rows)

print("=" * 65)
print("BLOCK 5 — DIVERSITY & NOVELTY")
print("=" * 65)

if not DIVERSITY_PREVIEW.empty:

    display(
        DIVERSITY_PREVIEW.groupby("model").agg(
            runs=("run_id", "count"),
            mean_diversity=("internal_diversity", "mean"),
            diversity_sd=("internal_diversity", "std"),
            mean_novelty_pct=("novelty_pct", "mean"),
            mean_unique=("unique_canonical", "mean")
        ).round(4)
    )

print("\nInterpretation:")
print("Diversity: higher means greater structural variation.")
print("Novelty: N/A unless a verified reference is configured.")
print("No production files modified.")


BLOCK 5 — DIVERSITY & NOVELTY


,runs,mean_diversity,diversity_sd,mean_novelty_pct,mean_unique
model,,,,,
finetuned_cdgcn_to_ctddg,20,0.8461,0.0146,NaN,100.0



Interpretation:
Diversity: higher means greater structural variation.
Novelty: N/A unless a verified reference is configured.
No production files modified.


## Random Forest Ames-mutagenicity evaluation

In [6]:
# ============================================================
# NOTEBOOK 3 — BLOCK 6
# RF AMES PREDICTOR AUDIT + OPTIONAL PREVIEW
# ============================================================

import json
import re
import joblib
import pandas as pd
import numpy as np

from pathlib import Path
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

# ------------------------------------------------------------
# A. RF CONFIGURATION
# ------------------------------------------------------------

RF_MODEL_PATH = (
    REWORKED_ROOT / "Random_Forest/RF_NItesh.joblib"
)

RF_RADIUS = 2
RF_BITS = 2048
RF_POSITIVE_LABEL = 1

# Set True only AFTER confirming from the RF training code:
# 1. Morgan bit fingerprints
# 2. radius = 2
# 3. 2048 bits
# 4. class 1 = Ames-mutagenic
#
# Leave False for this first audit.

RF_FEATURE_CONTRACT_VERIFIED = False

# Only load joblib/pickle artifacts you created or trust.
TRUST_LOCAL_RF_MODEL = True

# ------------------------------------------------------------
# B. DISCOVER RF FILES
# ------------------------------------------------------------

print("=" * 65)
print("RF MODEL DISCOVERY")
print("=" * 65)

rf_folder = REWORKED_ROOT / "Random_Forest"

print("Configured model:", RF_MODEL_PATH)
print("Model exists:", RF_MODEL_PATH.is_file())

if rf_folder.is_dir():

    model_files = [
        p for p in rf_folder.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {
            ".joblib", ".pkl", ".pickle"
        }
    ]

    print("\nSaved RF/model artifacts:")

    for path in sorted(model_files)[:20]:
        print(" ", path)

# ------------------------------------------------------------
# C. FIND TRAINING FEATURE DEFINITIONS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("RF FEATURE CONTRACT DISCOVERY")
print("=" * 65)

patterns = re.compile(
    r"Morgan|fingerprint|fpSize|nBits|radius|"
    r"Ames|mutagen|RandomForest|predict_proba|"
    r"GetFingerprint|ConvertToNumpyArray",
    re.IGNORECASE
)

source_files = []

if rf_folder.is_dir():
    source_files = sorted([
        p for p in rf_folder.rglob("*")
        if p.is_file()
        and p.suffix.lower() in {".py", ".ipynb"}
        and p.stat().st_size < 15 * 1024**2
    ])

print("Training code files:", len(source_files))

found = []

for path in source_files:

    try:
        if path.suffix.lower() == ".ipynb":

            notebook = json.loads(path.read_text())
            lines = []

            for cell in notebook.get("cells", []):
                if cell.get("cell_type") == "code":
                    src = cell.get("source", [])
                    code = (
                        "".join(src)
                        if isinstance(src, list) else src
                    )
                    lines.extend(code.splitlines())

        else:
            lines = path.read_text(
                errors="replace"
            ).splitlines()

        for i, line in enumerate(lines):

            if patterns.search(line):
                found.append({
                    "file": str(path),
                    "line": i + 1,
                    "code": "\n".join(lines[
                        max(0, i - 1):
                        min(len(lines), i + 3)
                    ])
                })

    except Exception as exc:
        print("Inspection error:", path, exc)

for record in found[:35]:

    print(
        f"\n{Path(record['file']).name}"
        f" : line {record['line']}"
    )
    print(record["code"][:500])

if len(found) > 35:
    print(f"\n... {len(found) - 35} more matches")

# ------------------------------------------------------------
# D. PREDICTION FUNCTION
# ------------------------------------------------------------

def predict_rf_ames(canonical_smiles, classifier):

    smiles = sorted(set(canonical_smiles))

    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=RF_RADIUS,
        fpSize=RF_BITS
    )

    predictions = {}

    batch_size = 1024

    for start in range(0, len(smiles), batch_size):

        portion = smiles[start:start + batch_size]

        X = np.zeros(
            (len(portion), RF_BITS),
            dtype=np.uint8
        )

        for i, sm in enumerate(portion):

            mol = Chem.MolFromSmiles(sm)
            assert mol is not None

            fingerprint = generator.GetFingerprint(mol)

            DataStructs.ConvertToNumpyArray(
                fingerprint, X[i]
            )

        labels = classifier.predict(X)

        for sm, label in zip(portion, labels):
            predictions[sm] = int(
                label == RF_POSITIVE_LABEL
            )

    return predictions

# ------------------------------------------------------------
# E. SCORE VERIFIED GROUPS IF CONTRACT IS VERIFIED
# ------------------------------------------------------------

rf_predictions = {}
RF_AMES_PREVIEW = pd.DataFrame()

if not RF_FEATURE_CONTRACT_VERIFIED:

    print("\n" + "=" * 65)
    print("RF INFERENCE NOT YET ENABLED")
    print("=" * 65)

    print(
        "Inspect the feature/label definitions above."
    )
    print(
        "Once verified, set "
        "RF_FEATURE_CONTRACT_VERIFIED = True "
        "and rerun Block 6."
    )

else:

    assert TRUST_LOCAL_RF_MODEL
    assert RF_MODEL_PATH.is_file(), RF_MODEL_PATH

    classifier = joblib.load(RF_MODEL_PATH)

    assert hasattr(classifier, "predict")
    assert hasattr(classifier, "classes_")

    print("\nRF classes:", classifier.classes_)

    assert RF_POSITIVE_LABEL in classifier.classes_, (
        "Configured mutagenic class is absent"
    )

    if hasattr(classifier, "n_features_in_"):
        assert classifier.n_features_in_ == RF_BITS, (
            "RF input dimension does not match"
        )

    # Read only the small preview groups.
    all_canonical = set()
    preview_groups = []

    for item in QA_VERIFIED_FILES:

        df = pd.read_csv(
            item["main_path"],
            keep_default_na=False
        )

        for run_id in range(1, EXPECTED_RUNS + 1):

            group = df[df.Run_ID == run_id]

            valid = [
                canonicalize_smiles(sm)
                for sm in group.SMILES
            ]

            valid = [sm for sm in valid if sm is not None]

            all_canonical.update(valid)

            preview_groups.append({
                "model": item["model"],
                "fold": item["fold"],
                "protein_id": item["protein_id"],
                "run_id": run_id,
                "canonical": valid,
            })

    print(
        "\nUnique molecules requiring RF prediction:",
        len(all_canonical)
    )

    rf_predictions = predict_rf_ames(
        all_canonical, classifier
    )

    rows = []

    for group in preview_groups:

        labels = [
            rf_predictions[sm]
            for sm in group["canonical"]
        ]

        toxic = sum(labels)
        count = len(labels)

        rows.append({
            "model": group["model"],
            "fold": group["fold"],
            "protein_id": group["protein_id"],
            "run_id": group["run_id"],
            "scored": count,
            "predicted_mutagenic": toxic,
            "rf_ames_mutagenic_pct": (
                100 * toxic / count if count else np.nan
            ),
        })

    RF_AMES_PREVIEW = pd.DataFrame(rows)

    print("\n" + "=" * 65)
    print("RF AMES — PRELIMINARY RESULTS")
    print("=" * 65)

    display(
        RF_AMES_PREVIEW.groupby("model").agg(
            runs=("run_id", "count"),
            molecules_scored=("scored", "sum"),
            mean_ames_mutagenic_pct=(
                "rf_ames_mutagenic_pct", "mean"
            ),
            predicted_mutagenic=(
                "predicted_mutagenic", "sum"
            )
        ).round(3)
    )

print("\nBLOCK 6 COMPLETE.")
print("No generated molecule files modified.")


RF MODEL DISCOVERY
Configured model: /home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/RF_NItesh.joblib
Model exists: True

Saved RF/model artifacts:
  /home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/RF_NItesh.joblib
  /home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/not_needed/rf_outputs/best_ames_rf.joblib
  /home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/not_needed/rf_outputs/new_hansen_rf.joblib

RF FEATURE CONTRACT DISCOVERY
Training code files: 1

RF.ipynb : line 8
# -------- REQUIRED INPUTS: EDIT THESE FOUR PATHS --------
HANSEN_N7090 = Path('/home/muhamed/repo/CTDDG/datasets/Mutagenicity_N7090.csv')  # tab-separated
HANSEN_N6512 = Path('/home/muhamed/repo/CTDDG/datasets/Mutagenicity_N6512.csv')  # comma-separated
CHEMBL_CSV = Path('/home/muhamed/repo/CTDDG/datasets/data/chembl/chembl.csv')

RF.ipynb : line 9
HANSEN_N7090 = Path('/home/muhamed/repo/CTDDG/datasets/Mutagenicity_N7090.csv')  # tab-separated
HANSEN_N6512 = Path('/home/muhamed/repo/CTDDG/datasets/Mut

## Audit and select the Ames RF model

In [8]:

# ============================================================
# NOTEBOOK 3 — BLOCK 7
# VERIFY RANDOM FOREST MODEL PROVENANCE
# ============================================================

from pathlib import Path
import json
import joblib
import pandas as pd

RF_DIR = REWORKED_ROOT / "Random_Forest"
RF_OUTPUT_DIR = RF_DIR / "not_needed/rf_outputs"

RF_CANDIDATES = {
    "old_nitesh": RF_DIR / "RF_NItesh.joblib",
    "new_hansen": RF_OUTPUT_DIR / "new_hansen_rf.joblib",
    "selected_best": RF_OUTPUT_DIR / "best_ames_rf.joblib",
}

RF_MODEL_AUDIT = {}

# Only use joblib files from your own trusted runs.
TRUST_LOCAL_RF_FILES = True
assert TRUST_LOCAL_RF_FILES

print("=" * 65)
print("1. SAVED MODEL COMPARISON")
print("=" * 65)

for name, path in RF_CANDIDATES.items():

    if not path.is_file():
        print("MISSING:", name, path)
        continue

    model = joblib.load(path)

    RF_MODEL_AUDIT[name] = {
        "file": str(path),
        "type": type(model).__name__,
        "features": getattr(model, "n_features_in_", None),
        "classes": list(
            getattr(model, "classes_", [])
        ),
        "estimators": getattr(model, "n_estimators", None),
        "sha256": sha256_file(path),
    }

display(pd.DataFrame.from_dict(
    RF_MODEL_AUDIT, orient="index"
))

# ------------------------------------------------------------
# 2. INSPECT MODEL-SELECTION METADATA
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("2. MODEL SELECTION METADATA")
print("=" * 65)

metadata_files = sorted(
    RF_OUTPUT_DIR.glob("*.json")
) if RF_OUTPUT_DIR.is_dir() else []

for path in metadata_files:

    print("\nFILE:", path)

    try:
        data = json.loads(path.read_text())
        print(json.dumps(
            data, indent=2, default=str
        )[:5500])
    except Exception as exc:
        print("Unable to read:", exc)

if not metadata_files:
    print("No selection metadata JSON found.")

comparison_path = (
    RF_OUTPUT_DIR / "rf_comparison_metrics.csv"
)

if comparison_path.is_file():
    print("\nRF validation comparison:")
    display(pd.read_csv(comparison_path))

# ------------------------------------------------------------
# 3. INSPECT LABEL SEMANTICS AND MODEL SELECTION CODE
# ------------------------------------------------------------


# ============================================================
# BLOCK 7 — SECTION 3 (FIXED)
# DISCOVER RF TRAINING NOTEBOOKS RECURSIVELY
# ============================================================

print("=" * 65)
print("3. RF TRAINING NOTEBOOK — LABEL DEFINITIONS")
print("=" * 65)

notebook_files = sorted(
    p for p in RF_DIR.rglob("RF.ipynb")
    if p.is_file()
)

print("Training notebooks found:", len(notebook_files))

keywords = [
    "activity",
    "mutagenic",
    "non-mutagenic",
    "positive",
    "label_map",
    "selected_rf",
    "selected_model",
    "best_model",
    "joblib.dump",
    "new_rf.fit",
    "old_rf",
    "selection",
]

matches = []

for notebook_path in notebook_files:

    print("\nInspecting:", notebook_path)

    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    for cell_number, cell in enumerate(
        notebook["cells"], start=1
    ):

        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", [])
        code = (
            "".join(source)
            if isinstance(source, list)
            else source
        )

        lines = code.splitlines()

        for index, line in enumerate(lines):

            if any(
                keyword.lower() in line.lower()
                for keyword in keywords
            ):

                matches.append({
                    "file": str(notebook_path),
                    "cell": cell_number,
                    "line": index + 1,
                    "code": "\n".join(
                        lines[
                            max(0, index - 2):
                            min(len(lines), index + 3)
                        ]
                    ),
                })

if not notebook_files:
    print(
        "No RF.ipynb found recursively. "
        "Search for renamed training notebooks."
    )

print("\nMatching training-code references:", len(matches))

for item in matches[:45]:

    print(
        f"\n{Path(item['file']).name} | "
        f"Cell {item['cell']} | "
        f"Line {item['line']}"
    )

    print(item["code"][:650])

if len(matches) > 45:
    print(f"\n... {len(matches) - 45} additional matches")

print("\nBLOCK 7 COMPLETE")
print("No predictions calculated.")


1. SAVED MODEL COMPARISON


/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,file,type,features,classes,estimators,sha256
old_nitesh,/home/muhamed/repo/CTDDG/REWORKED_CODE/Random_...,RandomForestClassifier,2048,"[0.0, 1.0]",100,69ca3802c519ceea4e0d4d4e30ab3e8031d0f2ed4e8584...
new_hansen,/home/muhamed/repo/CTDDG/REWORKED_CODE/Random_...,RandomForestClassifier,2048,"[0, 1]",500,a01a8e8d2f16ae02f57e47719d459c601286495a61d8ad...
selected_best,/home/muhamed/repo/CTDDG/REWORKED_CODE/Random_...,RandomForestClassifier,2048,"[0.0, 1.0]",100,88eb0e097a966100cb5ecda988051e4095d3c85d6f92b0...



2. MODEL SELECTION METADATA

FILE: /home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/not_needed/rf_outputs/best_ames_rf_metadata.json
{
  "selected_model": "Previous RF (loaded)",
  "selection_metric": "ROC-AUC",
  "selected_model_score": 0.9776445768677011,
  "selected_fingerprint": {
    "kind": "morgan",
    "radius": 2,
    "n_bits": 2048
  },
  "new_model_fingerprint": {
    "kind": "morgan",
    "radius": 2,
    "n_bits": 2048
  },
  "new_rf_config": {
    "n_estimators": 500,
    "random_state": 42,
    "n_jobs": -1
  },
  "hyperparameter_tuning": "not performed",
  "previous_model_original_path": "/home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/RF_NItesh.joblib",
  "test_fraction": 0.2,
  "random_seed": 42,
  "held_out_test_samples": 1417,
  "comparison_metrics": [
    {
      "Model": "New RF (trained here)",
      "Accuracy": 0.803105151729005,
      "Precision": 0.8253094910591472,
      "Recall": 0.7978723404255319,
      "F1": 0.8113590263691683,
      "ROC-AUC": 

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,Test samples
0,New RF (trained here),0.803105,0.825309,0.797872,0.811359,0.884516,1417
1,Previous RF (loaded),0.939308,0.938158,0.948138,0.943122,0.977645,1417


3. RF TRAINING NOTEBOOK — LABEL DEFINITIONS
Training notebooks found: 1

Inspecting: /home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/not_needed/RF.ipynb

Matching training-code references: 58

RF.ipynb | Cell 5 | Line 4

# -------- REQUIRED INPUTS: EDIT THESE FOUR PATHS --------
HANSEN_N7090 = Path('/home/muhamed/repo/CTDDG/datasets/Mutagenicity_N7090.csv')  # tab-separated
HANSEN_N6512 = Path('/home/muhamed/repo/CTDDG/datasets/Mutagenicity_N6512.csv')  # comma-separated
CHEMBL_CSV = Path('/home/muhamed/repo/CTDDG/datasets/data/chembl/chembl.csv')

RF.ipynb | Cell 5 | Line 5
# -------- REQUIRED INPUTS: EDIT THESE FOUR PATHS --------
HANSEN_N7090 = Path('/home/muhamed/repo/CTDDG/datasets/Mutagenicity_N7090.csv')  # tab-separated
HANSEN_N6512 = Path('/home/muhamed/repo/CTDDG/datasets/Mutagenicity_N6512.csv')  # comma-separated
CHEMBL_CSV = Path('/home/muhamed/repo/CTDDG/datasets/data/chembl/chembl.csv')
OLD_RF_JOBLIB = Path('/home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/RF_NI

## Ames prediction smoke test

In [10]:
# ============================================================
# FIX — INITIALIZE VERIFIED AMES RF
# Run immediately BEFORE Block 8
# ============================================================

from pathlib import Path
import joblib
import hashlib

RF_EVALUATION_MODEL = (
    REWORKED_ROOT
    / "Random_Forest/not_needed/rf_outputs"
    / "new_hansen_rf.joblib"
)

assert RF_EVALUATION_MODEL.is_file(), RF_EVALUATION_MODEL

# Only load your own trusted joblib model.
rf_model = joblib.load(RF_EVALUATION_MODEL)

# Confirm compatibility with the training notebook.
assert rf_model.n_features_in_ == 2048
assert set(rf_model.classes_) == {0, 1}
assert hasattr(rf_model, "predict")

RF_RADIUS = 2
RF_BITS = 2048
RF_MUTAGENIC_CLASS = 1

RF_CONTRACT_VERIFIED = True

print("=" * 60)
print("AMES RF CONFIGURATION VERIFIED")
print("=" * 60)
print("Model:", RF_EVALUATION_MODEL.name)
print("Fingerprint: Morgan, radius 2, 2048 bits")
print("Classes:", rf_model.classes_)
print("Mutagenic class:", RF_MUTAGENIC_CLASS)
print("Feature contract verified:", RF_CONTRACT_VERIFIED)
print("SHA256:", hashlib.sha256(
    RF_EVALUATION_MODEL.read_bytes()
).hexdigest()[:20])
print("\nPASS — Ready to run Block 8.")


/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


AMES RF CONFIGURATION VERIFIED
Model: new_hansen_rf.joblib
Fingerprint: Morgan, radius 2, 2048 bits
Classes: [0 1]
Mutagenic class: 1
Feature contract verified: True
SHA256: a01a8e8d2f16ae02f57e

PASS — Ready to run Block 8.


In [11]:
import joblib
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

# ------------------------------------------------------------
# A. SETTINGS TO CONFIRM AFTER BLOCK 7
# ------------------------------------------------------------

# Choose ONE model for ALL five generators.
# Initially None until Block 7 is reviewed.
assert RF_CONTRACT_VERIFIED

# Example after verification:
# RF_EVALUATION_MODEL = RF_CANDIDATES["new_hansen"]

assert RF_EVALUATION_MODEL.is_file()

RF_RADIUS = 2
RF_BITS = 2048

# Only use 1 if training labels confirm that
# class 1 means Ames-mutagenic.
RF_MUTAGENIC_CLASS = 1

RF_PREDICTION_BATCH_SIZE = 2048

# ------------------------------------------------------------
# B. FINGERPRINT + PREDICTION FUNCTION
# ------------------------------------------------------------

def score_ames_rf(smiles_list, classifier):

    unique_smiles = sorted(set(smiles_list))

    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=RF_RADIUS,
        fpSize=RF_BITS
    )

    predictions = {}

    for start in range(
        0, len(unique_smiles), RF_PREDICTION_BATCH_SIZE
    ):

        batch = unique_smiles[
            start:start + RF_PREDICTION_BATCH_SIZE
        ]

        X = np.zeros(
            (len(batch), RF_BITS),
            dtype=np.uint8
        )

        for i, smiles in enumerate(batch):

            mol = Chem.MolFromSmiles(smiles)
            assert mol is not None

            fp = generator.GetFingerprint(mol)

            DataStructs.ConvertToNumpyArray(
                fp, X[i]
            )

        labels = classifier.predict(X)

        for smiles, label in zip(batch, labels):
            predictions[smiles] = int(
                label == RF_MUTAGENIC_CLASS
            )

    return predictions

# ------------------------------------------------------------
# C. VERIFY MODEL BEFORE PREDICTING
# ------------------------------------------------------------

RF_AMES_PREVIEW = pd.DataFrame()

if RF_EVALUATION_MODEL is None or not RF_CONTRACT_VERIFIED:

    print("=" * 65)
    print("AMES RF — CONFIGURATION PENDING")
    print("=" * 65)

    print("Block 7 must establish:")
    print("1. Which model was selected and why")
    print("2. Exact fingerprint representation")
    print("3. Feature count and Morgan radius")
    print("4. Which class means Ames-mutagenic")
    print()
    print("No predictions calculated.")

else:

    selected_path = Path(RF_EVALUATION_MODEL)

    assert selected_path.is_file()
    assert selected_path.resolve() in {
        p.resolve()
        for p in RF_CANDIDATES.values()
        if p.is_file()
    }

    classifier = joblib.load(selected_path)

    assert hasattr(classifier, "predict")
    assert RF_MUTAGENIC_CLASS in classifier.classes_
    assert classifier.n_features_in_ == RF_BITS

    # --------------------------------------------------------
    # D. COLLECT VERIFIED PREVIEW MOLECULES
    # --------------------------------------------------------

    unique_molecules = set()
    groups = []

    for item in QA_VERIFIED_FILES:

        df = pd.read_csv(
            item["main_path"],
            keep_default_na=False
        )

        for run_id in range(1, EXPECTED_RUNS + 1):

            group = df[df.Run_ID == run_id]

            canonical = [
                canonicalize_smiles(s)
                for s in group.SMILES
            ]

            valid = [
                s for s in canonical
                if s is not None
            ]

            unique_molecules.update(valid)

            groups.append({
                "model": item["model"],
                "fold": item["fold"],
                "protein_id": item["protein_id"],
                "run_id": run_id,
                "valid": valid,
                "exported_count": len(group),
            })

    # --------------------------------------------------------
    # E. RF INFERENCE
    # --------------------------------------------------------

    predictions = score_ames_rf(
        unique_molecules, classifier
    )

    results = []

    for group in groups:

        labels = [
            predictions[s] for s in group["valid"]
        ]

        n_scored = len(labels)
        n_mutagenic = sum(labels)

        results.append({
            "model": group["model"],
            "fold": group["fold"],
            "protein_id": group["protein_id"],
            "run_id": group["run_id"],
            "exported_count": group["exported_count"],
            "rf_scored_count": n_scored,
            "rf_mutagenic_count": n_mutagenic,
            "rf_coverage_pct": (
                100 * n_scored / group["exported_count"]
                if group["exported_count"] else np.nan
            ),
            "rf_ames_mutagenic_pct": (
                100 * n_mutagenic / n_scored
                if n_scored else np.nan
            ),
        })

    RF_AMES_PREVIEW = pd.DataFrame(results)

    print("=" * 65)
    print("RF AMES MUTAGENICITY — PREVIEW")
    print("=" * 65)

    print("RF model:", selected_path)
    print("Unique molecules scored:", len(predictions))

    display(
        RF_AMES_PREVIEW.groupby("model").agg(
            runs=("run_id", "count"),
            molecules_scored=("rf_scored_count", "sum"),
            mutagenic=("rf_mutagenic_count", "sum"),
            mean_mutagenic_pct=(
                "rf_ames_mutagenic_pct", "mean"
            ),
            mean_coverage_pct=(
                "rf_coverage_pct", "mean"
            )
        ).round(3)
    )

print("\nBLOCK 8 COMPLETE.")


/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/muhamed/.conda/envs/ctddg_torch/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.3.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


RF AMES MUTAGENICITY — PREVIEW
RF model: /home/muhamed/repo/CTDDG/REWORKED_CODE/Random_Forest/not_needed/rf_outputs/new_hansen_rf.joblib
Unique molecules scored: 2000


,runs,molecules_scored,mutagenic,mean_mutagenic_pct,mean_coverage_pct
model,,,,,
finetuned_cdgcn_to_ctddg,20,2000,189,9.45,100.0



BLOCK 8 COMPLETE.


## RF prediction reproducibility

In [12]:

# ============================================================
# NOTEBOOK 3 — BLOCK 9
# RF REPRODUCIBILITY ACROSS SCIKIT-LEARN VERSIONS
# ============================================================

import os
import sys
import json
import hashlib
import subprocess
import textwrap
from pathlib import Path
import numpy as np
import pandas as pd

assert RF_CONTRACT_VERIFIED
assert RF_EVALUATION_MODEL.is_file()
assert QA_VERIFIED_FILES
assert RF_RADIUS == 2
assert RF_BITS == 2048
assert RF_MUTAGENIC_CLASS == 1

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Adjust only if Conda created this environment elsewhere.
RF_PINNED_PYTHON = (
    Path.home()
    / ".conda/envs/ctddg_rf132/bin/python"
)

# This worker will also be reused by Block 10.
RF_WORKER = RESULTS_DIR / "_rf_inference_worker.py"

worker_code = textwrap.dedent(r'''
import sys, os, json, hashlib
from pathlib import Path

import joblib
import sklearn
import numpy as np
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import rdFingerprintGenerator

input_path, model_path, output_path = map(Path, sys.argv[1:4])

with input_path.open() as f:
    smiles = json.load(f)["smiles"]

assert smiles == sorted(set(smiles))
assert smiles

model = joblib.load(model_path)

assert model.n_features_in_ == 2048
assert list(model.classes_) == [0, 1]
assert hasattr(model, "predict_proba")

generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048
)

labels = []
probabilities = []
feature_hash = hashlib.sha256()

for start in range(0, len(smiles), 2048):
    batch = smiles[start:start + 2048]
    X = np.zeros((len(batch), 2048), dtype=np.uint8)

    for i, sm in enumerate(batch):
        mol = Chem.MolFromSmiles(sm)
        if mol is None:
            raise ValueError(f"Invalid molecule: {sm}")

        fp = generator.GetFingerprint(mol)
        DataStructs.ConvertToNumpyArray(fp, X[i])

    feature_hash.update(X.tobytes())

    pred = model.predict(X)
    prob = model.predict_proba(X)

    labels.extend(int(v) for v in pred)
    probabilities.extend(float(v) for v in prob[:, 1])

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

result = {
    "model_sha256": sha(model_path),
    "input_sha256": sha(input_path),
    "fingerprint_sha256": feature_hash.hexdigest(),
    "sklearn_version": sklearn.__version__,
    "rdkit_version": rdBase.rdkitVersion,
    "numpy_version": np.__version__,
    "classes": [0, 1],
    "radius": 2,
    "bits": 2048,
    "count": len(smiles),
    "labels": labels,
    "positive_probabilities": probabilities,
}

output_path.parent.mkdir(parents=True, exist_ok=True)
tmp = output_path.with_suffix(".tmp")
tmp.write_text(json.dumps(result))
os.replace(tmp, output_path)

print("Scored:", len(smiles))
print("scikit-learn:", sklearn.__version__)
print("Predicted mutagenic:", sum(labels))
''')

if RF_WORKER.exists():
    assert RF_WORKER.read_text() == worker_code, (
        "Existing RF worker differs. Review before replacing."
    )
else:
    RF_WORKER.write_text(worker_code)

# ------------------------------------------------------------
# A. USE EXACTLY THE SAME PREVIEW MOLECULES
# ------------------------------------------------------------

preview_smiles = set()

for item in QA_VERIFIED_FILES:
    df = pd.read_csv(item["main_path"], keep_default_na=False)

    for sm in df["SMILES"]:
        canonical = canonicalize_smiles(sm)
        if canonical is not None:
            preview_smiles.add(canonical)

preview_smiles = sorted(preview_smiles)

payload = json.dumps(
    {"smiles": preview_smiles},
    separators=(",", ":")
)

preview_id = hashlib.sha256(
    payload.encode()
).hexdigest()[:16]

RF_REPRO_DIR = RESULTS_DIR / "_rf_repro" / preview_id
RF_REPRO_DIR.mkdir(parents=True, exist_ok=True)

RF_REPRO_INPUT = RF_REPRO_DIR / "preview_input.json"
RF_REPRO_CURRENT = RF_REPRO_DIR / "current_result.json"
RF_REPRO_PINNED = RF_REPRO_DIR / "sklearn132_result.json"

if RF_REPRO_INPUT.exists():
    assert RF_REPRO_INPUT.read_text() == payload
else:
    RF_REPRO_INPUT.write_text(payload)

def run_rf_worker(python_executable, output_path):
    command = [
        str(python_executable),
        str(RF_WORKER),
        str(RF_REPRO_INPUT),
        str(RF_EVALUATION_MODEL),
        str(output_path)
    ]

    completed = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if completed.returncode != 0:
        print(completed.stderr[-3000:])
        raise RuntimeError(
            f"RF inference failed: {python_executable}"
        )

    print(completed.stdout.strip())
    return json.loads(output_path.read_text())

# ------------------------------------------------------------
# B. COMPARE ENVIRONMENTS
# ------------------------------------------------------------

RF_REPRO_VERIFIED = False

print("=" * 65)
print("RF PREDICTION REPRODUCIBILITY")
print("=" * 65)

print("Molecules:", len(preview_smiles))
print("Current Python:", sys.executable)
print("Pinned Python:", RF_PINNED_PYTHON)

current_result = run_rf_worker(
    sys.executable, RF_REPRO_CURRENT
)

if RF_PINNED_PYTHON.is_file():

    pinned_result = run_rf_worker(
        RF_PINNED_PYTHON, RF_REPRO_PINNED
    )

    assert pinned_result["sklearn_version"] == "1.3.2"
    assert pinned_result["model_sha256"] == current_result["model_sha256"]
    assert pinned_result["input_sha256"] == current_result["input_sha256"]

    same_fingerprints = (
        pinned_result["fingerprint_sha256"]
        == current_result["fingerprint_sha256"]
    )

    same_labels = (
        pinned_result["labels"]
        == current_result["labels"]
    )

    current_probs = np.array(
        current_result["positive_probabilities"]
    )
    pinned_probs = np.array(
        pinned_result["positive_probabilities"]
    )

    max_probability_difference = float(
        np.max(np.abs(current_probs - pinned_probs))
    )

    probabilities_match = bool(
        np.allclose(
            current_probs, pinned_probs,
            atol=1e-10, rtol=0
        )
    )

    RF_REPRO_VERIFIED = bool(
        same_fingerprints
        and same_labels
        and probabilities_match
    )

    display(pd.DataFrame([{
        "same_fingerprints": same_fingerprints,
        "same_labels": same_labels,
        "probabilities_match": probabilities_match,
        "max_probability_difference":
            max_probability_difference,
        "reproducible": RF_REPRO_VERIFIED
    }]))

    if not RF_REPRO_VERIFIED:
        raise RuntimeError(
            "RF predictions differ across versions. "
            "Use pinned sklearn 1.3.2 for evaluation."
        )

    print("\nPASS — RF reproducibility check completed.")

else:
    print("\nPinned environment not found.")
    print("Create ctddg_rf132 using the terminal command above.")
    print("Then rerun Block 9.")

print("RF_REPRO_VERIFIED:", RF_REPRO_VERIFIED)


RF PREDICTION REPRODUCIBILITY
Molecules: 2000
Current Python: /home/muhamed/.conda/envs/ctddg_torch/bin/python
Pinned Python: /home/muhamed/.conda/envs/ctddg_rf132/bin/python
Scored: 2000
scikit-learn: 1.9.1
Predicted mutagenic: 189

Pinned environment not found.
Create ctddg_rf132 using the terminal command above.
Then rerun Block 9.
RF_REPRO_VERIFIED: False


## FULL DATASET EVALUATION + 10-RUN STATISTICS

In [13]:

# ============================================================
# NOTEBOOK 3 — BLOCK 10
# FULL DATASET EVALUATION + 10-RUN STATISTICS
# ============================================================

import os
import json
import hashlib
import subprocess
from pathlib import Path
from collections import Counter, defaultdict

import h5py
import numpy as np
import pandas as pd
from rdkit import rdBase

# ------------------------------------------------------------
# A. REQUIRE VERIFIED CONFIGURATION
# ------------------------------------------------------------

import sys
import sklearn

assert RF_CONTRACT_VERIFIED
assert RF_EVALUATION_MODEL.is_file()
assert RF_WORKER.is_file()

RF_INFERENCE_PYTHON = Path(sys.executable)

print("Using existing environment:", RF_INFERENCE_PYTHON)
print("scikit-learn:", sklearn.__version__)
print("Cross-version RF verification: NOT PERFORMED")

assert EXPECTED_RUNS == 10
assert EXPECTED_PER_RUN == 100

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Correct the older CDGCN checkpoint path when
# switching to all-five evaluation later.
if EVALUATION_MODE == "all5":
    MODEL_SPECS["older_pretrained_no_toxicity"][
        "checkpoint"
    ] = (
        REWORKED_ROOT
        / "pretraining/pretraining_runs/"
        "cdgcn_torch_nll_bs32_v1/checkpoint.pt"
    )

# Each cache is tied to input hashes and the
# RDKit implementation used for the metrics.
METRIC_REVISION = "nb3_core_metrics_v1"
CACHE_ROOT = RESULTS_DIR / "_verified_metric_cache"

model_hashes = {}

for name in ACTIVE_MODELS:
    spec = MODEL_SPECS[name]
    assert spec["checkpoint"].is_file(), spec["checkpoint"]
    assert spec["vocab"].is_file(), spec["vocab"]

    model_hashes[name] = {
        "checkpoint": sha256_file(spec["checkpoint"]),
        "vocab": sha256_file(spec["vocab"]),
    }

# ------------------------------------------------------------
# B. COMPLETE-CORPUS PREFLIGHT
# ------------------------------------------------------------

input_groups = []
missing = []

for model_name in ACTIVE_MODELS:
    spec = MODEL_SPECS[model_name]
    model_root = MODEL_INPUT_ROOTS[spec["phase"]] / model_name

    for fold, manifest in FOLD_MANIFESTS.items():
        for row in manifest.itertuples(index=False):

            folder = model_root / fold / row.protein_id

            accepted_path = (
                folder / f"{row.protein_id}_all_samples.csv"
            )
            raw_path = (
                folder / f"{row.protein_id}_raw_attempts.csv"
            )
            info_path = folder / "generation_info.json"

            required = [
                accepted_path, raw_path, info_path
            ] + [
                folder / "runs" / f"run_{i:02d}"
                / "run_info.json"
                for i in range(1, EXPECTED_RUNS + 1)
            ]

            absent = [p for p in required if not p.is_file()]

            if absent:
                missing.append({
                    "model": model_name,
                    "fold": fold,
                    "protein": row.protein_id,
                    "missing": str(absent[0])
                })
            else:
                input_groups.append({
                    "model": model_name,
                    "fold": fold,
                    "protein_id": row.protein_id,
                    "sequence_sha256": row.sequence_sha256,
                    "cleaned_order_id": int(row.cleaned_order_id),
                    "folder": folder,
                    "accepted": accepted_path,
                    "raw": raw_path,
                    "info": info_path
                })

expected_groups = (
    EXPECTED_SLOTS_PER_MODEL * len(ACTIVE_MODELS)
)

print("=" * 65)
print("FULL EVALUATION PREFLIGHT")
print("=" * 65)

print("Expected protein/model groups:", expected_groups)
print("Complete group files found:", len(input_groups))
print("Missing/incomplete groups:", len(missing))

if missing:
    display(pd.DataFrame(missing).head(15))
    raise RuntimeError(
        "Generation is incomplete. Finish Notebook 2, "
        "then rerun Block 10."
    )

assert len(input_groups) == expected_groups

print("PASS — All expected groups have saved output files.")

# ------------------------------------------------------------
# C. UTILITIES
# ------------------------------------------------------------

def pct(n, d):
    return 100.0 * n / d if d else None

def atomic_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, indent=2, allow_nan=False))
    os.replace(tmp, path)

def atomic_dataframe(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)

# ------------------------------------------------------------
# D. EVALUATE EACH PROTEIN, SAVING PROGRESS
# ------------------------------------------------------------

metric_rows = []
canonical_by_group = {}
all_unique_canonical = set()
unique_by_model = defaultdict(set)

for group_index, item in enumerate(input_groups, 1):

    model = item["model"]
    fold = item["fold"]
    protein = item["protein_id"]
    folder = item["folder"]

    info = json.loads(item["info"].read_text())

    assert info["model"] == model
    assert info["fold"] == fold
    assert info["protein_id"] == protein
    assert info["protein_sequence_sha256"] == item[
        "sequence_sha256"
    ]
    assert info["conditioned_on_protein"] == (
        MODEL_SPECS[model]["conditioned"]
    )
    assert info["checkpoint_sha256"] == (
        model_hashes[model]["checkpoint"]
    )
    assert info["vocabulary_sha256"] == (
        model_hashes[model]["vocab"]
    )
    assert info["runs"] == EXPECTED_RUNS
    assert info["target_per_run"] == EXPECTED_PER_RUN
    assert info["accepted_molecules"] == EXPECTED_PER_SLOT

    if MODEL_SPECS[model]["conditioned"]:
        embedding_path = (
            EMBEDDINGS_ROOT / fold
            / "prose_embeddings_avg_6165.h5"
        )

        with h5py.File(embedding_path, "r") as h5:
            vector = np.asarray(
                h5[f"p{item['cleaned_order_id']}"],
                dtype=np.float32
            )

        vector_sha = hashlib.sha256(
            vector.tobytes()
        ).hexdigest()

        assert info["protein_embedding_sha256"] == vector_sha
    else:
        assert info["protein_embedding_sha256"] is None

    input_shas = {
        "accepted": sha256_file(item["accepted"]),
        "raw": sha256_file(item["raw"]),
        "metadata": sha256_file(item["info"]),
    }

    assert input_shas["accepted"] == info["samples_sha256"]
    assert input_shas["raw"] == info["raw_attempts_sha256"]

    cache_path = (
        CACHE_ROOT / model / fold / f"{protein}.json"
    )

    expected_cache_identity = {
        "revision": METRIC_REVISION,
        "rdkit_version": rdBase.rdkitVersion,
        "input_shas": input_shas,
        "checkpoint_sha256": model_hashes[model]["checkpoint"],
    }

    if cache_path.is_file():
        cache = json.loads(cache_path.read_text())

        for key, value in expected_cache_identity.items():
            assert cache.get(key) == value, (
                "Outdated/corrupt evaluation cache: "
                f"{cache_path}. Review it before restarting."
            )

        group_metric_rows = cache["metric_rows"]
        group_canonical = cache["canonical_by_run"]

    else:
        accepted = pd.read_csv(
            item["accepted"], keep_default_na=False
        )
        raw = pd.read_csv(
            item["raw"], keep_default_na=False
        )

        assert {"Run_ID", "SMILES"}.issubset(accepted.columns)
        assert {
            "Run_ID", "Attempt_ID", "SMILES",
            "decode_status", "saved_in_main_csv"
        }.issubset(raw.columns)

        assert len(accepted) == EXPECTED_PER_SLOT
        assert len(raw) == info["raw_attempts"]

        flags_text = (
            raw["saved_in_main_csv"]
            .astype(str).str.lower()
        )
        assert flags_text.isin(
            ["true", "false", "1", "0"]
        ).all()

        flags = flags_text.isin(["true", "1"])

        assert Counter(zip(
            raw.loc[flags, "Run_ID"],
            raw.loc[flags, "SMILES"]
        )) == Counter(zip(
            accepted["Run_ID"],
            accepted["SMILES"]
        )), f"{model}/{fold}/{protein}: raw flags mismatch"

        group_metric_rows = []
        group_canonical = {}

        for run_id in range(1, EXPECTED_RUNS + 1):

            a = accepted[accepted.Run_ID == run_id]
            r = raw[raw.Run_ID == run_id]

            assert len(a) == EXPECTED_PER_RUN
            assert a.SMILES.nunique() == EXPECTED_PER_RUN
            assert len(r) >= EXPECTED_PER_RUN

            assert r.Attempt_ID.tolist() == list(
                range(1, len(r) + 1)
            )

            run_dir = folder / "runs" / f"run_{run_id:02d}"
            run_info = json.loads(
                (run_dir / "run_info.json").read_text()
            )

            assert run_info["run_id"] == run_id
            assert run_info["accepted"] == EXPECTED_PER_RUN
            assert run_info["attempts"] == len(r)

            assert sha256_file(
                run_dir / "accepted.csv"
            ) == run_info["accepted_sha256"]

            assert sha256_file(
                run_dir / "raw_attempts.csv"
            ) == run_info["raw_sha256"]

            canonical = [
                canonicalize_smiles(sm)
                for sm in a.SMILES
            ]
            valid = [
                sm for sm in canonical if sm is not None
            ]

            raw_valid = [
                canonicalize_smiles(sm) is not None
                for sm in r.SMILES
            ]

            ended_valid = sum(
                v and str(status).lower() == "ended"
                for v, status in zip(
                    raw_valid, r.decode_status
                )
            )

            props = pd.DataFrame([
                molecular_properties(sm) for sm in valid
            ])

            unique_raw = a.SMILES.nunique()
            unique_canonical = set(valid)

            row = {
                "model": model,
                "fold": fold,
                "protein_id": protein,
                "run_id": run_id,
                "exported_count": len(a),
                "raw_attempt_count": len(r),

                "mentor_validity_pct":
                    pct(len(a), EXPECTED_PER_RUN),

                "raw_rdkit_validity_pct":
                    pct(sum(raw_valid), len(r)),

                "raw_ended_valid_pct":
                    pct(ended_valid, len(r)),

                "exported_rdkit_validity_pct":
                    pct(len(valid), len(a)),

                "mentor_uniqueness_pct":
                    pct(unique_raw, EXPECTED_PER_RUN),

                "canonical_uniqueness_valid_pct":
                    pct(len(unique_canonical), len(valid)),

                "QED_mean_valid":
                    float(props.QED.mean())
                    if len(valid) else None,

                "SAS_mean_valid":
                    float(props.SAS.mean())
                    if len(valid) else None,

                "mentor_QED":
                    float(props.QED.sum() / EXPECTED_PER_RUN)
                    if len(valid) else None,
            }

            tests = {
                "logP_lt5": props.logP < 5,
                "MW_lt500": props.MW < 500,
                "HBD_lt5": props.HBD < 5,
                "HBA_lt10": props.HBA < 10,
                "RB_lt10": props.RB < 10,
                "TPSA_lt140": props.TPSA < 140,
                "SAS_lt6": props.SAS < 6,
            } if len(valid) else {}

            for metric in [
                "logP_lt5", "MW_lt500", "HBD_lt5",
                "HBA_lt10", "RB_lt10",
                "TPSA_lt140", "SAS_lt6"
            ]:
                passing = int(tests[metric].sum()) if tests else 0

                row[f"{metric}_standard_pct"] = (
                    pct(passing, len(valid))
                )
                row[f"{metric}_mentor_pct"] = (
                    100.0
                    * unique_raw / EXPECTED_PER_RUN
                    * passing / EXPECTED_PER_RUN
                )

            if len(valid):
                violations = (
                    (props.logP >= 5).astype(int)
                    + (props.MW >= 500).astype(int)
                    + (props.HBD >= 5).astype(int)
                    + (props.HBA >= 10).astype(int)
                )

                row["lipinski_zero_violations_pct"] = pct(
                    int((violations == 0).sum()),
                    len(valid)
                )
            else:
                row["lipinski_zero_violations_pct"] = None

            seed_text = f"{model}|{fold}|{protein}|{run_id}"
            seed = int(
                hashlib.sha256(seed_text.encode())
                .hexdigest()[:8], 16
            )

            diversity, pair_count = (
                calculate_internal_diversity(
                    list(unique_canonical),
                    seed + DIVERSITY_SEED
                )
            )

            row["internal_diversity"] = (
                float(diversity)
                if np.isfinite(diversity) else None
            )
            row["diversity_pairs"] = int(pair_count)

            group_metric_rows.append(row)
            group_canonical[str(run_id)] = valid

        atomic_json({
            **expected_cache_identity,
            "metric_rows": group_metric_rows,
            "canonical_by_run": group_canonical,
        }, cache_path)

    assert len(group_metric_rows) == EXPECTED_RUNS

    metric_rows.extend(group_metric_rows)

    for run_id in range(1, EXPECTED_RUNS + 1):
        key = (model, fold, protein, run_id)
        smiles = group_canonical[str(run_id)]

        canonical_by_group[key] = smiles
        all_unique_canonical.update(smiles)
        unique_by_model[model].update(smiles)

    if group_index % 25 == 0 or group_index == len(input_groups):
        print(
            f"Evaluated {group_index}/{len(input_groups)} "
            "protein/model groups",
            flush=True
        )

print("\nAll molecular descriptors computed.")
print("Unique canonical molecules:", len(all_unique_canonical))

# ------------------------------------------------------------
# E. FULL RF INFERENCE WITH PINNED SCIKIT-LEARN
# ------------------------------------------------------------

unique_smiles = sorted(all_unique_canonical)

rf_payload = json.dumps(
    {"smiles": unique_smiles},
    separators=(",", ":")
)

rf_input_hash = hashlib.sha256(
    rf_payload.encode()
).hexdigest()

rf_model_hash = sha256_file(RF_EVALUATION_MODEL)

RF_FULL_ROOT = RESULTS_DIR / "_rf_full"
RF_FULL_ROOT.mkdir(parents=True, exist_ok=True)

RF_FULL_INPUT = (
    RF_FULL_ROOT
    / f"input_{rf_input_hash[:16]}.json"
)
RF_FULL_OUTPUT = (
    RF_FULL_ROOT
    / f"predictions_sklearn_{sklearn.__version__}_"
      f"{rf_input_hash[:16]}_{rf_model_hash[:12]}.json"
)

if RF_FULL_INPUT.exists():
    assert RF_FULL_INPUT.read_text() == rf_payload
else:
    RF_FULL_INPUT.write_text(rf_payload)

if not RF_FULL_OUTPUT.exists():

    command = [
        str(RF_INFERENCE_PYTHON),
        str(RF_WORKER),
        str(RF_FULL_INPUT),
        str(RF_EVALUATION_MODEL),
        str(RF_FULL_OUTPUT)
    ]

    print("\nRunning full Ames RF inference...", flush=True)

    completed = subprocess.run(
        command, capture_output=True, text=True
    )

    if completed.returncode != 0:
        print(completed.stderr[-4000:])
        raise RuntimeError("Full RF inference failed.")

    print(completed.stdout)

rf_result = json.loads(RF_FULL_OUTPUT.read_text())

assert rf_result["sklearn_version"] == sklearn.__version__
assert rf_result["input_sha256"] == sha256_file(RF_FULL_INPUT)
assert rf_result["model_sha256"] == rf_model_hash
assert rf_result["count"] == len(unique_smiles)
assert len(rf_result["labels"]) == len(unique_smiles)
assert set(rf_result["labels"]).issubset({0, 1})

ames_map = dict(zip(unique_smiles, rf_result["labels"]))

# ------------------------------------------------------------
# F. NOVELTY + AMES RATES PER PROTEIN/RUN
# ------------------------------------------------------------

for row in metric_rows:

    key = (
        row["model"], row["fold"],
        row["protein_id"], row["run_id"]
    )

    smiles = canonical_by_group[key]

    labels = [ames_map[s] for s in smiles]

    row["rf_ames_scored"] = len(labels)
    row["rf_ames_positive"] = int(sum(labels))
    row["rf_ames_mutagenic_pct"] = pct(
        sum(labels), len(labels)
    )

    reference = REFERENCE_SETS.get(row["model"])

    if reference is None:
        row["novelty_pct"] = None
    else:
        unique = set(smiles)
        row["novelty_pct"] = pct(
            len(unique - reference), len(unique)
        )

# ------------------------------------------------------------
# G. RUN-LEVEL AGGREGATION
# ------------------------------------------------------------

PER_PROTEIN_RUN = pd.DataFrame(metric_rows)

metric_columns = [
    "mentor_validity_pct",
    "raw_rdkit_validity_pct",
    "raw_ended_valid_pct",
    "exported_rdkit_validity_pct",
    "mentor_uniqueness_pct",
    "canonical_uniqueness_valid_pct",
    "QED_mean_valid",
    "SAS_mean_valid",
    "mentor_QED",
    "internal_diversity",
    "novelty_pct",
    "rf_ames_mutagenic_pct",
    "lipinski_zero_violations_pct",
] + [
    f"{name}_{suffix}_pct"
    for name in [
        "logP_lt5", "MW_lt500", "HBD_lt5",
        "HBA_lt10", "RB_lt10",
        "TPSA_lt140", "SAS_lt6"
    ]
    for suffix in ["standard", "mentor"]
]

PER_RUN = (
    PER_PROTEIN_RUN
    .groupby(["model", "run_id"], as_index=False)
    .agg({
        **{m: "mean" for m in metric_columns},
        "protein_id": "count",
        "exported_count": "sum",
        "raw_attempt_count": "sum",
        "rf_ames_positive": "sum",
        "rf_ames_scored": "sum",
    })
    .rename(columns={"protein_id": "protein_slots"})
)

assert all(
    PER_RUN.loc[
        PER_RUN.model == name, "run_id"
    ].nunique() == EXPECTED_RUNS
    for name in ACTIVE_MODELS
)

overall_rows = []

for model in ACTIVE_MODELS:

    runs = PER_RUN[PER_RUN.model == model]
    groups = PER_PROTEIN_RUN[
        PER_PROTEIN_RUN.model == model
    ]

    summary = {
        "model": model,
        "protein_conditioned":
            MODEL_SPECS[model]["conditioned"],
        "runs": len(runs),
        "protein_slots": groups[
            ["fold", "protein_id"]
        ].drop_duplicates().shape[0],
        "accepted_molecules": int(
            groups.exported_count.sum()
        ),
        "raw_attempts": int(
            groups.raw_attempt_count.sum()
        ),
        "distinct_canonical_molecules":
            len(unique_by_model[model]),
    }

    for metric in metric_columns:
        values = pd.to_numeric(
            runs[metric], errors="coerce"
        ).dropna()

        summary[metric + "_mean"] = (
            float(values.mean()) if len(values) else np.nan
        )
        summary[metric + "_sd"] = (
            float(values.std(ddof=0))
            if len(values) == EXPECTED_RUNS else np.nan
        )

    overall_rows.append(summary)

OVERALL_COMPARISON = pd.DataFrame(overall_rows)

# ------------------------------------------------------------
# H. SAVE FINAL REPORTS
# ------------------------------------------------------------

atomic_dataframe(
    PER_PROTEIN_RUN,
    RESULTS_DIR / "per_protein_run_metrics.csv"
)

atomic_dataframe(
    PER_RUN,
    RESULTS_DIR / "per_run_aggregates.csv"
)

atomic_dataframe(
    OVERALL_COMPARISON,
    RESULTS_DIR / "overall_model_comparison.csv"
)

provenance = {
    "evaluation_mode": EVALUATION_MODE,
    "model_names": ACTIVE_MODELS,
    "checkpoint_hashes": model_hashes,
    "protein_slots_per_model": EXPECTED_SLOTS_PER_MODEL,
    "runs": EXPECTED_RUNS,
    "samples_per_run": EXPECTED_PER_RUN,
    "rf_model": str(RF_EVALUATION_MODEL),
    "rf_model_sha256": rf_model_hash,
    "rf_sklearn_version": rf_result["sklearn_version"],
    "rf_cross_version_reproducibility_verified": False,
    "rf_version_compatibility": "Unverified; interpret Ames predictions cautiously",
    "rf_fingerprint_sha256":
        rf_result["fingerprint_sha256"],
    "rf_predictions_input_sha256": rf_input_hash,
    "rdkit_version": rdBase.rdkitVersion,
    "metric_revision": METRIC_REVISION,
    "novelty_reference_models": list(REFERENCE_SETS),
    "population_sd_across_run_means": True,
    "unconditional_model_protein_ids_are_sampling_slots": True,
}

atomic_json(
    provenance,
    RESULTS_DIR / "evaluation_provenance.json"
)

# ------------------------------------------------------------
# I. DISPLAY ESSENTIAL COMPARISON
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FULL EVALUATION — RESULTS")
print("=" * 65)

display(OVERALL_COMPARISON[[
    "model",
    "accepted_molecules",
    "raw_rdkit_validity_pct_mean",
    "mentor_uniqueness_pct_mean",
    "internal_diversity_mean",
    "QED_mean_valid_mean",
    "SAS_mean_valid_mean",
    "rf_ames_mutagenic_pct_mean",
]].round(4))

print("\nResults saved under:", RESULTS_DIR)
print("NOTEBOOK 3 — BLOCK 10 COMPLETE")


Using existing environment: /home/muhamed/.conda/envs/ctddg_torch/bin/python
scikit-learn: 1.9.1
Cross-version RF verification: NOT PERFORMED
FULL EVALUATION PREFLIGHT
Expected protein/model groups: 606
Complete group files found: 606
Missing/incomplete groups: 0
PASS — All expected groups have saved output files.
Evaluated 25/606 protein/model groups
Evaluated 50/606 protein/model groups
Evaluated 75/606 protein/model groups
Evaluated 100/606 protein/model groups
Evaluated 125/606 protein/model groups
Evaluated 150/606 protein/model groups
Evaluated 175/606 protein/model groups
Evaluated 200/606 protein/model groups
Evaluated 225/606 protein/model groups
Evaluated 250/606 protein/model groups
Evaluated 275/606 protein/model groups
Evaluated 300/606 protein/model groups
Evaluated 325/606 protein/model groups
Evaluated 350/606 protein/model groups
Evaluated 375/606 protein/model groups
Evaluated 400/606 protein/model groups
Evaluated 425/606 protein/model groups
Evaluated 450/606 protei

,model,accepted_molecules,raw_rdkit_validity_pct_mean,mentor_uniqueness_pct_mean,internal_diversity_mean,QED_mean_valid_mean,SAS_mean_valid_mean,rf_ames_mutagenic_pct_mean
0,finetuned_cdgcn_to_ctddg,202000,95.5415,100.0,0.8458,0.4812,3.2992,5.7436
1,pretrained_no_toxicity,202000,97.9583,100.0,0.8861,0.6597,2.7900,19.7085
2,pretrained_with_toxicity,202000,59.3431,100.0,0.8712,0.6126,3.1115,1.0248



Results saved under: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/metrics/phase1_evaluation_v1
NOTEBOOK 3 — BLOCK 10 COMPLETE
